In [1]:
import json
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path
import time
import gc

from rich.progress import Progress

from python_magnetrun.MagnetRun import MagnetRun, load_mrun
from python_magnetrun.signature import Signature

In [2]:
DATA_DIR = Path("../Data")
DB = Path("../to_duckdb/magnetdb.duckdb")

FIELD_THRESHOLD = 0.1

# HOUSING SUMMARY
### Load and merge the housing summary files


In [3]:
PATH_COLUMNS = [
    "overview", "archive", "pupitre", "default", "trigger", "spike",
    "hybrid_kHz", "hybrid_rms", "hybrid_trigger", "hybrid_vprocess",
    "pigbrother_runlog", "pupitre_runlog",
]

rows = []
for file in sorted(DATA_DIR.glob("*_summary-*.json")):

    print(f"Loading {file.name}")

    housing = file.stem.split("_")[0]
    year = int(file.stem[-4: ])

    with open(file, "r") as f:
        data = json.load(f)

    df = pd.json_normalize(data)

    for col in PATH_COLUMNS:
        df[col] = df[col].apply(lambda x: Path(x).name if x else x)

    df["housing"] = housing
    df["year"] = year

    rows.append(df)

summary_df = pd.concat(rows, ignore_index = True)

summary_df["experiment_id"]       = None
summary_df["field_max"]           = pd.Series(dtype = "float64")
summary_df["field_mean"]          = pd.Series(dtype = "float64")
summary_df["field_time_on"]       = pd.Series(dtype = "float64")
summary_df["mode"]                = ""
summary_df["field_signature"]     = ""
summary_df["reference_signature"] = ""

print(f"Found {len(summary_df)} summary files")
print(summary_df.head())

Loading M10_summary-2022.json
Loading M10_summary-2023.json
Loading M10_summary-2024.json
Loading M10_summary-2025.json
Loading M10_summary-2026.json
Loading M9_summary-2022.json
Loading M9_summary-2023.json
Loading M9_summary-2024.json
Loading M9_summary-2025.json
Loading M9_summary-2026.json
Found 1776 summary files
                   filename                       overview  \
0  M10_Overview_220204-1003  M10_Overview_220204-1003.tdms   
1  M10_Overview_220204-1510  M10_Overview_220204-1510.tdms   
2  M10_Overview_220206-1521  M10_Overview_220206-1521.tdms   
3  M10_Overview_220208-0951  M10_Overview_220208-0951.tdms   
4  M10_Overview_220211-0941  M10_Overview_220211-0941.tdms   

                        archive                    pupitre default trigger  \
0  M10_Archive_220204-1003.tdms  2022.02.04 - 10:04:03.txt                   
1  M10_Archive_220204-1510.tdms  2022.02.04 - 15:10:08.txt                   
2  M10_Archive_220206-1521.tdms  2022.02.06 - 15:21:53.txt               

### Refresh the housing summary table and display basic stats

In [4]:
con = duckdb.connect(DB)

con.execute(
    """
        DROP TABLE IF EXISTS housing_summary
    """
)
con.register("summary_df", summary_df)
con.execute(
    """
        CREATE TABLE housing_summary AS
        SELECT *
        FROM summary_df
    """
)

print("\nRows:\n",
    con.execute(
        """
            SELECT housing, year, COUNT(*) AS n
            FROM housing_summary
            GROUP BY housing, year
            ORDER BY housing, year
        """
    ).fetchdf()
)


##DEBUG
rows = con.execute(
    """
        SELECT housing, pupitre
        FROM housing_summary
        WHERE pupitre <> ''
    """
).fetchall()
found = False
for housing, pupitre in rows:
    try:
        md = load_mrun(Path(pupitre).name, housing = housing).getMData()
    except FileNotFoundError:
        continue
    print(pupitre)
    found = True
    break
if not found:
    raise RuntimeError("No existing Pupitre file found.")
##DEBUG


Rows:
   housing  year    n
0     M10  2022  238
1     M10  2023  209
2     M10  2024  146
3     M10  2025  233
4     M10  2026   66
5      M9  2022  237
6      M9  2023  222
7      M9  2024  162
8      M9  2025  191
9      M9  2026   72


2022.02.04 - 10:04:03.txt


### Data quality audit

In [5]:
## Check the date schema
print("\nTABLE SCHEMA: ",
    con.execute(
        """
            DESCRIBE housing_summary
        """
    ).fetchdf()
)
## Count imported records
print("\nNUMBER OF ROWS:", 
    con.execute(
        """
            SELECT COUNT(*) FROM housing_summary
        """
    ).fetchone()[0]
)
# Count number of records linked to experiments
print("NUMBER OF MATCHES:",
    con.execute(
        """
            SELECT COUNT(*)
            FROM housing_summary AS h
            JOIN experiments AS e
            ON h.pupitre LIKE '%' || e.file
        """).fetchone()[0]
)


TABLE SCHEMA:              column_name column_type null   key default extra
0              filename     VARCHAR  YES  None    None  None
1              overview     VARCHAR  YES  None    None  None
2               archive     VARCHAR  YES  None    None  None
3               pupitre     VARCHAR  YES  None    None  None
4               default     VARCHAR  YES  None    None  None
5               trigger     VARCHAR  YES  None    None  None
6                 spike     VARCHAR  YES  None    None  None
7            hybrid_kHz     VARCHAR  YES  None    None  None
8            hybrid_rms     VARCHAR  YES  None    None  None
9        hybrid_trigger     VARCHAR  YES  None    None  None
10      hybrid_vprocess     VARCHAR  YES  None    None  None
11    pigbrother_runlog     VARCHAR  YES  None    None  None
12       pupitre_runlog     VARCHAR  YES  None    None  None
13              housing     VARCHAR  YES  None    None  None
14                 year      BIGINT  YES  None    None  None
15      

NUMBER OF MATCHES: 463


In [6]:
# Check for missing files
print("\nMISSING FILES:\n",
    con.execute(
        """
            SELECT
                SUM(CASE WHEN overview = '' THEN 1 ELSE 0 END) AS overview,
                SUM(CASE WHEN archive  = '' THEN 1 ELSE 0 END) AS archive,
                SUM(CASE WHEN pupitre  = '' THEN 1 ELSE 0 END) AS pupitre
            from housing_summary
        """
    ).fetchdf()
)
# Check for duplicate files
print("\nDUPLICATE FILENAMES:\n",
    con.execute(
        """
            SELECT filename, COUNT(*) AS n
            FROM housing_summary
            GROUP BY filename
            HAVING COUNT(*) > 1
            ORDER BY n DESC
        """
    ).fetchdf()
)

# List records for which no Pupitre file is available
print(
    con.execute(
        """
            SELECT filename, housing, year, pupitre
            FROM housing_summary
            WHERE pupitre IS NULL OR pupitre = ''
            ORDER BY year, housing, filename
        """
    ).fetchdf()
)


MISSING FILES:
    overview  archive  pupitre
0       0.0     23.0    149.0

DUPLICATE FILENAMES:
 Empty DataFrame
Columns: [filename, n]
Index: []
                     filename housing  year pupitre
0    M10_Overview_220420-1126     M10  2022        
1    M10_Overview_220711-1059     M10  2022        
2    M10_Overview_220711-1102     M10  2022        
3    M10_Overview_220720-0021     M10  2022        
4    M10_Overview_220720-0028     M10  2022        
..                        ...     ...   ...     ...
144   M9_Overview_260116-1559      M9  2026        
145   M9_Overview_260120-1052      M9  2026        
146   M9_Overview_260205-1752      M9  2026        
147   M9_Overview_260415-1012      M9  2026        
148   M9_Overview_260505-1130      M9  2026        

[149 rows x 4 columns]


### Link with the user DB: Add foreign key column and populate

In [7]:
con.execute(
    """
        UPDATE housing_summary AS h
        SET experiment_id = e.id
        FROM experiments AS e
        WHERE h.pupitre LIKE '%' || e.file
    """
)

print("\nLINKED EXPERIMENTS:",
    con.execute(
        """
            SELECT COUNT(*)
            FROM housing_summary
            WHERE experiment_id IS NOT NULL
        """
    ).fetchone()[0]
)
print(
    con.execute(
        """
            SELECT experiment_id, filename, pupitre
            FROM housing_summary
            WHERE experiment_id IS NOT NULL
            LIMIT 10
        """
    ).fetchdf()
)
print(
    con.execute(
        """
            SELECT h.experiment_id, e.name, e.file, h.pupitre
            FROM housing_summary AS h
            JOIN experiments AS e
            ON h.experiment_id = e.id
            LIMIT 10
        """
    ).fetchdf()
)

rows = con.execute(
    """
        SELECT rowid, housing, pupitre
        FROM housing_summary
        WHERE pupitre <> ''
    """
).fetchall()


LINKED EXPERIMENTS: 463
   experiment_id                  filename                    pupitre
0           1249  M10_Overview_250127-1605  2025.01.27 - 16:33:39.txt
1            756  M10_Overview_250313-1523  2025.03.13 - 15:14:35.txt
2            757  M10_Overview_250313-1525  2025.03.13 - 15:31:07.txt
3            759  M10_Overview_250313-1539  2025.03.13 - 15:44:41.txt
4            760  M10_Overview_250315-1509  2025.03.15 - 15:09:06.txt
5            764  M10_Overview_250315-1524  2025.03.15 - 16:53:01.txt
6            765  M10_Overview_250316-1331  2025.03.16 - 13:31:37.txt
7            766  M10_Overview_250316-2059  2025.03.16 - 20:59:53.txt
8            767  M10_Overview_250317-0936  2025.03.17 - 09:36:55.txt
9            768  M10_Overview_250408-1736  2025.04.08 - 17:36:07.txt
   experiment_id                   name                       file  \
0            756  2025.03.13 - 15:14:35  2025.03.13 - 15:14:35.txt   
1            757  2025.03.13 - 15:31:07  2025.03.13 - 15:31:07.tx

In [8]:
print(
    con.execute("""
        DESCRIBE housing_summary
    """).fetchdf()
)

            column_name column_type null   key default extra
0              filename     VARCHAR  YES  None    None  None
1              overview     VARCHAR  YES  None    None  None
2               archive     VARCHAR  YES  None    None  None
3               pupitre     VARCHAR  YES  None    None  None
4               default     VARCHAR  YES  None    None  None
5               trigger     VARCHAR  YES  None    None  None
6                 spike     VARCHAR  YES  None    None  None
7            hybrid_kHz     VARCHAR  YES  None    None  None
8            hybrid_rms     VARCHAR  YES  None    None  None
9        hybrid_trigger     VARCHAR  YES  None    None  None
10      hybrid_vprocess     VARCHAR  YES  None    None  None
11    pigbrother_runlog     VARCHAR  YES  None    None  None
12       pupitre_runlog     VARCHAR  YES  None    None  None
13              housing     VARCHAR  YES  None    None  None
14                 year      BIGINT  YES  None    None  None
15        experiment_id 

In [9]:
start = time.perf_counter()

with Progress() as progress:

    task = progress.add_task("Updating field stats", total=len(rows))
    
    for rowid, housing, pupitre in rows:
        filename = Path(pupitre).name
        progress.update(task, description=f"{housing}/{filename}")

        try:
            md = load_mrun(filename, housing = housing).getMData()
            df = md.Data

            signature = Signature.from_mdata(md, "Field", "t", FIELD_THRESHOLD)
            field_signature = json.dumps({"times": signature.times, "values": signature.values})
            field = df["Field"]

            con.execute(
                """
                    UPDATE housing_summary
                    SET 
                        field_max = ?,
                        field_mean = ?,
                        field_time_on = ?,
                        field_signature = ?
                    WHERE rowid = ?
                """, 
                (float(field.max()), float(field.mean()), int((field > FIELD_THRESHOLD).sum()), field_signature, int(rowid))
            )

            del signature
            del field
            del df
            del md
            del field_signature
            gc.collect()

        except Exception as e:
            progress.console.print(f"[red]{filename}: {e}[/red]")

        progress.advance(task)

end = time.perf_counter()
print(f"Dataframe updated in {int((end - start) // 60)} m {((end - start) % 60):.2f} s")

Output()

Duplicates found in _timestamp: 
/home/LNCMI-G/christophe.trophime/LNCMIG-Data/records/srv-data-install/M10/2022.02.12 - 14:20:02.txt — 1 
duplicate(s) removed

PandasMagnetData: dropping duplicate Icoil columns ['Icoil6'] from 
'/home/LNCMI-G/christophe.trophime/LNCMIG-Data/records/srv-data-install/M10/2022.03.31 - 14:29:32.txt'

PandasMagnetData: dropping duplicate Icoil columns ['Icoil6'] from 
'/home/LNCMI-G/christophe.trophime/LNCMIG-Data/records/srv-data-install/M10/2022.03.31 - 14:29:32.txt'

PandasMagnetData: dropping duplicate Icoil columns ['Icoil6'] from 
'/home/LNCMI-G/christophe.trophime/LNCMIG-Data/records/srv-data-install/M10/2022.03.31 - 14:29:32.txt'

...

Duplicates found in _timestamp: 
/home/LNCMI-G/christophe.trophime/LNCMIG-Data/records/srv-data-install/M10/2023.10.27 - 17:14:26.txt — 1 
duplicate(s) removed

PandasMagnetData: dropping duplicate Icoil columns ['Icoil3'] from 
'/home/LNCMI-G/christophe.trophime/LNCMIG-Data/records/srv-data-install/M10/2024.04.05 - 11:15:47.txt'

Duplicates found in _timestamp: 
/home/LNCMI-G/christophe.trophime/LNCMIG-Data/records/srv-data-install/M10/2024.05.04 - 15:22:14.txt — 1 
duplicate(s) removed

Duplicates found in _timestamp: 
/home/LNCMI-G/christophe.trophime/LNCMIG-Data/records/srv-data-install/M10/2024.05.21 - 15:20:56.txt — 1 
duplicate(s) removed

Duplicates found in _timestamp: 
/home/LNCMI-G/christophe.trophime/LNCMIG-Data/records/srv-data-install/M10/2024.05.24 - 17:16:40.txt — 1 
duplicate(s) removed

In [10]:
###
print(con.execute("SELECT COUNT(*) FROM housing_summary").fetchone())
con.execute("""
SELECT
    MIN(field_max),
    MAX(field_max),
    COUNT(field_max),
    COUNT(NULLIF(field_signature, ''))
FROM housing_summary
""").fetchdf()

(1776,)


,min(field_max),max(field_max),count(field_max),"count(""nullif""(field_signature, ''))"
0,0.0,38.044,1627,1627


In [11]:
# Validate update
print(
    con.execute(
        """
            SELECT COUNT(field_max) AS field_stats, COUNT(field_signature) AS signatures
            FROM housing_summary
        """
    ).fetchdf()
)
print(
    con.execute(
        """
            SELECT experiment_id, field_max, field_signature
            FROM housing_summary
            WHERE field_signature <> ''
            LIMIT 10
        """
    ).fetchdf()
)

print(
    con.execute(
        """
            SELECT experiment_id, field_signature
            FROM housing_summary
            WHERE field_signature IS NOT NULL
            LIMIT 5
        """
    ).fetchdf()
)

   field_stats  signatures
0         1627        1776
   experiment_id  field_max                    field_signature
0           <NA>     0.0043  {"times": [0.0], "values": [0.0]}
1           <NA>     6.9961  {"times": [0.0], "values": [0.0]}
2           <NA>     6.9962  {"times": [0.0], "values": [0.0]}
3           <NA>     6.9962  {"times": [0.0], "values": [0.0]}
4           <NA>     6.9962  {"times": [0.0], "values": [0.0]}
5           <NA>     6.9962  {"times": [0.0], "values": [0.0]}
6           <NA>     6.9962  {"times": [0.0], "values": [0.0]}
7           <NA>     6.9962  {"times": [0.0], "values": [0.0]}
8           <NA>     6.9962  {"times": [0.0], "values": [0.0]}
9           <NA>     6.9962  {"times": [0.0], "values": [0.0]}
   experiment_id                    field_signature
0           <NA>  {"times": [0.0], "values": [0.0]}
1           <NA>  {"times": [0.0], "values": [0.0]}
2           <NA>  {"times": [0.0], "values": [0.0]}
3           <NA>  {"times": [0.0], "values": 

In [12]:
for rowid, housing, pupitre in rows:
    filename = Path(pupitre).name

    try:
        md = load_mrun(filename, housing=housing)
    except FileNotFoundError:
        continue

    mdata = md.getMData()

    print("Testing:", filename)

    signature = Signature.from_mdata(
        mdata,
        "Field",
        "t",
        FIELD_THRESHOLD
    )

    print("Regimes:", len(signature.regimes))
    print("Times:", signature.times)
    print("Values:", signature.values)

    break

Testing: 2022.02.04 - 10:04:03.txt
Regimes: 1
Times: [0.0]
Values: [0.0]


# MODE INFERRING

In [13]:
mrun = load_mrun("M10_Overview_251201-0909.tdms", housing = "M10", site = "M10")
mdata = mrun.getMData()
print(mdata)

print(mdata.Data["Courants_Alimentations"].columns)

TdmsMagnetData(Type=<DataType.TDMS: 1>, Groups={'Courants_Alimentations': {'Courant_A1': OrderedDict({'wf_start_time': np.datetime64('2025-12-01T08:09:13.922483'), 'wf_start_offset': 0.5, 'wf_increment': 1.0, 'wf_samples': 31546, 'NI_ChannelName': 'Courant_A1', 'NI_UnitDescription': 'Volts', 'unit_string': 'Volts'}), 'Courant_A2': OrderedDict({'wf_start_time': np.datetime64('2025-12-01T08:09:13.922483'), 'wf_start_offset': 0.5, 'wf_increment': 1.0, 'wf_samples': 31546, 'NI_ChannelName': 'Courant_A2', 'NI_UnitDescription': 'Volts', 'unit_string': 'Volts'}), 'Courant_A3': OrderedDict({'wf_start_time': np.datetime64('2025-12-01T08:09:13.922483'), 'wf_start_offset': 0.5, 'wf_increment': 1.0, 'wf_samples': 31546, 'NI_ChannelName': 'Courant_A3', 'NI_UnitDescription': 'Volts', 'unit_string': 'Volts'}), 'Courant_A4': OrderedDict({'wf_start_time': np.datetime64('2025-12-01T08:09:13.922483'), 'wf_start_offset': 0.5, 'wf_increment': 1.0, 'wf_samples': 31546, 'NI_ChannelName': 'Courant_A4', 'NI_Un

# PROPOSALS

In [14]:
# Load proposals metadata and parse experiment date ranges

proposals_df = pd.read_csv(DATA_DIR / "proposals.csv")
proposals_df["Debut"] = pd.to_datetime(proposals_df["Debut"], errors = "coerce")
proposals_df["Fin"]   = pd.to_datetime(proposals_df["Fin"],   errors = "coerce")

In [15]:
# Connect to the database and recreate the proposals table

con = duckdb.connect(DB)

con.execute(
    """
        DROP TABLE IF EXISTS proposals
    """
)
con.register("proposals_df", proposals_df)
con.execute(
    """
        CREATE TABLE proposals AS
        SELECT * FROM proposals_df
    """
)

In [16]:
# Inspect imported proposal schema as well as the experiments table
 
print(
    con.execute(
        """
            DESCRIBE proposals
        """
    ).fetchdf()
)
print(
    con.execute(
        """
            SELECT * FROM proposals 
            LIMIT 10
        """
    ).fetchdf()
)

print(
    con.execute(
        """
            DESCRIBE experiments
        """
    )
)
print(
    con.execute(
        """
            SELECT * FROM experiments
            LIMIT 10
        """
    ).fetchdf()
)


        column_name   column_type null   key default extra
0           Acronym       VARCHAR  YES  None    None  None
1         ProjectID        BIGINT  YES  None    None  None
2      ResearchArea       VARCHAR  YES  None    None  None
3          Facility       VARCHAR  YES  None    None  None
4      ProposalType       VARCHAR  YES  None    None  None
5        accessMode        DOUBLE  YES  None    None  None
6        CallNumber        BIGINT  YES  None    None  None
7                id        BIGINT  YES  None    None  None
8   ExperimentState       VARCHAR  YES  None    None  None
9              Site       VARCHAR  YES  None    None  None
10    ShotsHourDone        DOUBLE  YES  None    None  None
11       EnergyUsed        DOUBLE  YES  None    None  None
12            Debut  TIMESTAMP_NS  YES  None    None  None
13              Fin  TIMESTAMP_NS  YES  None    None  None
       Acronym  ProjectID ResearchArea  Facility ProposalType  accessMode  \
0    GMS06-217       3218           MS

In [17]:
# Check temporal coverage of the proposal metadata

print(
    con.execute(
        """
            SELECT MIN(file), MAX(file), COUNT(*)
            FROM experiments
        """
    ).fetchdf()
)
print(
    con.execute(
        """
            SELECT DISTINCT year
            FROM housing_summary
            ORDER BY year
        """
    ).fetchdf()
)
print(
    con.execute(
        """
            SELECT MIN(Debut), MAX(Fin), COUNT(*)
            FROM proposals
        """
    ).fetchdf()
)

                                           min(file)  \
0  /mnt/LNCMIG-Data/records/srv-data-install/M10/...   

                   max(file)  count_star()  
0  2026.06.17 - 09:52:35.txt          1700  
   year
0  2022
1  2023
2  2024
3  2025
4  2026
  min(Debut)   max(Fin)  count_star()
0 2009-01-19 2023-10-27          1712


In [18]:
# Add proposal column to housing_summary unless it already exists

con.execute(
    """
        ALTER TABLE housing_summary
        ADD COLUMN IF NOT EXISTS proposal VARCHAR;
    """
)

# Link housing records to proposals by magnet site and experiment date

con.execute(
    """
        UPDATE housing_summary AS h
        SET proposal = p.Acronym
        FROM proposals AS p
        WHERE h.pupitre <> '' AND h.pupitre IS NOT NULL
            AND h.housing = regexp_replace(p.Site, '[ie]$', '')
            AND strptime(right(replace(h.pupitre, '.txt', ''), 19), '%y.%m.%d - %H:%M:%S')
        BETWEEN CAST(p.Debut AS TIMESTAMP) AND CAST(p.Fin AS TIMESTAMP);
    """
)

In [19]:
# Validate propsal linkage

print(
    con.execute(
        """
            SELECT COUNT(*) AS total, COUNT(proposal) AS linked
            FROM housing_summary;
        """
    ).fetchdf()
)

   total  linked
0   1776     676


In [20]:
con.close()

In [21]:
proposals_df = pd.read_csv(DATA_DIR / "proposals_2026-07-22.csv")
proposals_df["Experiment Start Date"] = pd.to_datetime(proposals_df["Experiment Start Date"], errors = "coerce")
proposals_df["Experiment End Date"]   = pd.to_datetime(proposals_df["Experiment End Date"], errors = "coerce")

print(proposals_df[["Acronym", "Magnet Sites", "Experiment Start Date", "Experiment End Date"]].head(), proposals_df.shape)

     Acronym  Magnet Sites Experiment Start Date Experiment End Date
0  GIS01-226           NaN            2026-10-20          2026-10-25
1        NaN           NaN                   NaT                 NaT
2        NaN           NaN                   NaT                 NaT
3  GIS02-126           NaN                   NaT                 NaT
4        NaN           NaN                   NaT                 NaT (1912, 13)


In [22]:
# Check Magent Sites in new proposals_2026-07-26.csv

print(proposals_df["Magnet Sites"].dtype)
print(len(proposals_df))
print(proposals_df["Magnet Sites"].notna().sum())

float64
1912
0
